# Analyzing and tuning a transmon qubit
We will showcase two methods (EPR amd LOM) to analyze the same design. Specifically, we will use here the `advanced` methods to run the simulations and analysis, which directly contorl renderers and external packages. Please refer to the tutorial notebooks 4.1 and 4.2 to follow the `suggested` flow to run the analysis.

## Index
#### Transmon design
1. Prepare the single transmon qubit layout in qiskit-metal. <br>

#### Transmon analysis using EPR method
1. Set-up and run a finite element simulate to extract the eigenmode. <br>
1. Display EM fields to inspect quality of the setup. <br>
1. Identify junction parameters for the EPR analysis. <br>
1. Run EPR analysis on single eigenmode. <br>
1. Get qubit freq and anharmonicity. <br>
1. Calculate EPR of substrate.  <br>

#### Transmon analysis using LOM method
1. Calculate the capacitance matrix. <br>
1. Execute analysis on extracted LOM. <br>

## Prerequisite
You need to have a working local installation of Ansys.<br>
Also you will need the following directives and inports.

In [1]:
%reload_ext autoreload
%autoreload 2

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings
import pyEPR as epr

# 1. Create the Qbit design

Fix the design dimensions that you intend to reflect in the design rendering. <br>
Note that the design size extends from the origin into the first quadrant.

In [2]:
design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = '2mm'
design.chips.main.size['size_y'] = '2mm'

gui = MetalGUI(design)

Create a single transmon in the center of the chip previously defined.

In [3]:
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket

design.delete_all_components()

q1 = TransmonPocket(design, 'Q1', options = dict(
    pad_width = '425 um', 
    pocket_height = '650um',
    connection_pads=dict(
        readout = dict(loc_W=+1,loc_H=+1, pad_width='200um')
    )))

gui.rebuild()
gui.autoscale()

# 2. Analyze the transmon using the Eigenmode-EPR method

In this section we will use a semi-manual (advanced) analysis flow. Please refer to tutorial 4.2 for the `suggested` method. As illustrated, the methods are equivalent, but the advanced method allows you to directly override some renderer-specific settings.

### Finite Element Eigenmode Analysis

#### Setup

Select the analysis you intend to run from the `qiskit_metal.analyses` collection.<br>
Select the design to analyze and the tool to use for any external simulation.

In [4]:
from qiskit_metal.analyses.quantization import EPRanalysis
eig_qb = EPRanalysis(design, "hfss")

For the Eigenmode simulation portion, you can either:
1. Use the `eig_qb` user-friendly methods (see tutorial 4.2)
2. Control directly the simulation tool from the tool's GUI (outside metal - see specific vendor instructions)
3. Use the renderer methods
In this section we show the advanced method (method 3).

The renderer can be reached from the analysis class. Let's give it a shorter alias.

In [5]:
hfss = eig_qb.sim.renderer

Now we connect to the tool using the unified command.

In [6]:
hfss.start()

INFO 03:34AM [connect_project]: Connecting to Ansys Desktop API...
INFO 03:34AM [load_ansys_project]: 	Opened Ansys App
INFO 03:34AM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
INFO 03:34AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/ymino/Documents/Ansoft/
	Project:   Project6
INFO 03:34AM [connect_design]: No active design found (or error getting active design).
INFO 03:34AM [connect]: 	 Connected to project "Project6". No design detected


True

The previous command is supposed to open ansys (if closed), create a new project and finally connect this notebook to it.

If for any reason the previous cell failed, please try the manual path described in the next three cells:
1. uncomment and execute only **one** of the lines in the first cell.
1. uncomment and execute the second cell.
1. uncomment and execute only **one** of the lines in the third cell.

In [7]:
# hfss.open_ansys()   # this opens Ansys 2021 R2 if present
# hfss.open_ansys(path_var='ANSYSEM_ROOT211')
# hfss.open_ansys(path='C:\\Program Files\\AnsysEM\\AnsysEM21.1\\Win64')
# hfss.open_ansys(path='../../../Program Files/AnsysEM/AnsysEM21.1/Win64'

In [8]:
# hfss.new_ansys_project()

In [9]:
# hfss.connect_ansys()
# hfss.connect_ansys('C:\\project_path\\', 'Project1')  # will open a saved project before linking the Jupyter session

#### Execute simulation and verify convergence

Create and activate an eigenmode design called "TransmonQubit".

In [10]:
hfss.activate_ansys_design("TransmonQubit", 'eigenmode')  # use new_ansys_design() to force creation of a blank design

03:34AM 33s WARNING [activate_ansys_design]: The design_name=TransmonQubit was not in active project.  Designs in active project are: 
[].  A new design will be added to the project.  
INFO 03:34AM [connect_design]: 	Opened active design
	Design:    TransmonQubit [Solution type: Eigenmode]
WARNING 03:34AM [connect_setup]: 	No design setup detected.
WARNING 03:34AM [connect_setup]: 	Creating eigenmode default setup.
INFO 03:34AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


Render the single qubit in Metal, called Q1, to "TransmonQubit" design in Ansys.

In [11]:
hfss.render_design(['Q1'], [])
# hfss.save_screenshot()

Set the convergence parameters and junction properties in the Ansys design. <br>
Then run the analysis and plot the convergence.

In [12]:
# Analysis properties
setup = hfss.pinfo.setup
setup.passes = 10
print(f"""
Number of eigenmodes to find             = {setup.n_modes}
Number of simulation passes              = {setup.passes}
Convergence freq max delta percent diff  = {setup.delta_f}
""")

pinfo = hfss.pinfo
pinfo.design.set_variable('Lj', '10 nH')
pinfo.design.set_variable('Cj', '0 fF')

setup.analyze()

INFO 03:34AM [analyze]: Analyzing setup Setup



Number of eigenmodes to find             = 1
Number of simulation passes              = 10
Convergence freq max delta percent diff  = 0.1



To plot the results you can use the `plot_convergences()` method from the `eig_qb.sim` object. The method will read the data from the variables local to the `eig_qb.sim` object, so we first need to assign the simulation results to these two variables. let's do both (assignment and plotting) in the next cell.

In [13]:
eig_qb.sim.convergence_t, eig_qb.sim.convergence_f, _ = hfss.get_convergences()
eig_qb.sim.plot_convergences()

03:35AM 48s INFO [get_f_convergence]: Saved convergences to c:\Users\ymino\Documents\GitHub\qiskit-metal\ymino\hfss_eig_f_convergence.csv


#### Plot the EM field for inspection
Display the Ansys modeler window and plot the E-field on the chip's surface.

In [14]:
hfss.modeler._modeler.ShowWindow()
hfss.plot_fields('main')
#hfss.save_screenshot()

Delete the newly created E-field plot to prepare for the next phase.

In [15]:
hfss.plot_ansys_delete(['Mag_E1'])

03:35AM 50s WARNING [plot_ansys_delete]: This method is deprecated. Change your scripts to use clear_fields()


### EPR Analysis
In the `suggested` (tutorial 4.2) flow, we would now prepare the setup using `eig_qb.setup` and run the analysis with `eig_qb.run_epr()`. Notice that this method requires previous set of the `eig_qb` variables `convergence_t` and `convergence_f` like we did a thee cells earlier.

However we here exemplify the advanced approach, which is Ansys-specific since it uses the pyEPR module methods directly.
#### Setup
Identify the non-linear (Josephson) junctions in the model. You will need to list the junctions in the epr setup.

In this case there's only one junction, namely 'jj'. Let's see what we need to change in the default setup.

In [16]:
pinfo = hfss.pinfo
pinfo.junctions['jj'] = {'Lj_variable': 'Lj', 'rect': 'JJ_rect_Lj_Q1_rect_jj', 
                             'line': 'JJ_Lj_Q1_rect_jj_',  'Cj_variable': 'Cj'}
pinfo.validate_junction_info() # Check that valid names of variables and objects have been supplied
pinfo.dissipative['dielectrics_bulk'] = ['main'] # Dissipative elements: specify

#### Execute the energy distribution analysis

Execute microwave analysis on eigenmode solutions.

In [17]:
eprd = epr.DistributedAnalysis(pinfo)

Design "TransmonQubit" info:
	# eigenmodes    1
	# variations    1


Find the electric and magnetic energy stored in the substrate and the system as a whole.

In [18]:
ℰ_elec = eprd.calc_energy_electric()
ℰ_elec_substrate = eprd.calc_energy_electric(None, 'main')
ℰ_mag = eprd.calc_energy_magnetic()

print(f"""
ℰ_elec_all       = {ℰ_elec}
ℰ_elec_substrate = {ℰ_elec_substrate}
EPR of substrate = {ℰ_elec_substrate / ℰ_elec * 100 :.1f}%

ℰ_mag    = {ℰ_mag}
""")


ℰ_elec_all       = 2.18387508314691e-24
ℰ_elec_substrate = 2.01136753932794e-24
EPR of substrate = 92.1%

ℰ_mag    = 4.34988046253761e-26



#### Run the EPR analysis

Perform EPR analysis for all modes and variations.

In [20]:
eprd.do_EPR_analysis()

# 4a. Perform Hamiltonian spectrum post-analysis, building on mw solutions using EPR
epra = epr.QuantumAnalysis(eprd.data_filename)
epra.analyze_all_variations(cos_trunc = 8, fock_trunc = 7)

# 4b. Report solved results
swp_variable = 'Lj' # suppose we swept an optimetric analysis vs. inductance Lj_alice
epra.plot_hamiltonian_results(swp_variable=swp_variable)
epra.report_results(swp_variable=swp_variable, numeric=True)

  options=pd.Series(get_instance_vars(self.options)),

WARNING 04:29AM [__init__]: <p>Error: <class 'IndexError'></p>
  result['Q_coupling'] = self.Qm_coupling[variation][self.Qm_coupling[variation].columns[junctions]][modes]#TODO change the columns to junctions

  result['Qs'] = self.Qs[variation][self.PM[variation].columns[junctions]][modes] #TODO change the columns to junctions




Variation 0  [1/1]
  previously analyzed ...

ANALYSIS DONE. Data saved to:

C:\data-pyEPR\Project6\TransmonQubit\2025-07-16 03-35-50.npz


	 Differences in variations:



 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
Variation 0

Starting the diagonalization
Finished the diagonalization
Pm_norm=
modes
0    1.017207
dtype: float64

Pm_norm idx =
     jj
0  True
*** P (participation matrix, not normlz.)
         jj
0  0.963945

*** S (sign-bit matrix)
   s_jj
0     1
*** P (participation matrix, normalized.)
      0.98

*** Chi matrix O1 PT (MHz)
    Diag is anharmonicity, off diag is full cross-Kerr.
       297

*** Chi matrix ND (MHz) 
       330

*** Frequencies O1 PT (MHz)
0    6063.548559
dtype: float64

*** Frequencies ND (MHz)
0    6048.271989
dtype: float64

*** Q_coupling
Empty DataFrame
Columns: []
Index: [0]


#### Mode frequencies (MHz)

###### Numerical diagonalization

Lj,10
0,6048.27


#### Kerr Non-linear coefficient table (MHz)

###### Numerical diagonalization

,,0
Lj,,
10,0,329.57


Release Ansys session

In [21]:
eig_qb.sim.close()

# 3. Analyze the transmon using the LOM method

In this section we will use a semi-manual (advanced) analysis flow. Please refer to tutorial 4.1 for the `suggested` method. As illustrated, the methods are equivalent, but the advanced method allows you to directly override some renderer-specific settings.

### Capacitance matrix extraction
#### Setup
Select the analysis you intend to run from the `qiskit_metal.analyses` collection.<br>
Select the design to analyze and the tool to use for any external simulation.

In [22]:
from qiskit_metal.analyses.quantization import LOManalysis
c1 = LOManalysis(design, "q3d")

For the capacitive simulation portion, you can either:
1. Use the `c1` user-friendly methods (see tutorial 4.1)
2. Control directly the simulation tool from the tool's GUI (outside metal - see specific vendor instructions)
3. Use the renderer methods
In this section we show the advanced method (method 3).

The renderer can be reached from the analysis class. Let's give it a shorter alias.

In [23]:
q3d = c1.sim.renderer

Now we connect to the simulation tool, similarly to what we have done for the eigenmode analysis.

In [24]:
q3d.start()

INFO 04:29AM [connect_project]: Connecting to Ansys Desktop API...
INFO 04:29AM [load_ansys_project]: 	Opened Ansys App
INFO 04:29AM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
INFO 04:29AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/ymino/Documents/Ansoft/
	Project:   Project6
INFO 04:29AM [connect_design]: 	Opened active design
	Design:    TransmonQubit [Solution type: Eigenmode]
INFO 04:29AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 04:29AM [connect]: 	Connected to project "Project6" and design "TransmonQubit" 😀 



True

If the simulator is already open, the line above will simply connect to the open session, project and design.

#### Execute simulation and verify convergence

Create and activate a q3d design called "TransmonQubit_q3d".

In [25]:
q3d.activate_ansys_design("TransmonQubit_q3d", 'capacitive')  # use new_ansys_design() to force creation of a blank design

04:29AM 51s WARNING [activate_ansys_design]: The design_name=TransmonQubit_q3d was not in active project.  Designs in active project are: 
['TransmonQubit'].  A new design will be added to the project.  
INFO 04:29AM [connect_design]: 	Opened active design
	Design:    TransmonQubit_q3d [Solution type: Q3D]
WARNING 04:29AM [connect_setup]: 	No design setup detected.
WARNING 04:29AM [connect_setup]: 	Creating Q3D default setup.
INFO 04:29AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)


Next, we render the qubit to Ansys Q3D for analysis. We set the readout pin of the qubit in the 'open' termination list of the render so its capacitance is properly simulated.

In [26]:
q3d.render_design(['Q1'], [('Q1','readout')])

Execute the capacitance extraction and verify converengence. This cell analyzes the default setup.

In [27]:
q3d.analyze_setup("Setup")

INFO 04:29AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 04:29AM [analyze]: Analyzing setup Setup


This simulation had 4 nets, the two charge islands of the floating transmon, the readout coupler, and the ground, resulting in a 4x4 capacitance matrix. Output is of type DataFrame.

In [28]:
c1.sim.capacitance_matrix, c1.sim.units = q3d.get_capacitance_matrix()
c1.sim.capacitance_all_passes, _ = q3d.get_capacitance_all_passes()
c1.sim.capacitance_matrix

INFO 04:30AM [get_matrix]: Exporting matrix data to (C:\Users\ymino\AppData\Local\Temp\5\tmpng25xacf.txt, C, , Setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 04:30AM [get_matrix]: Exporting matrix data to (C:\Users\ymino\AppData\Local\Temp\5\tmp0fdzjkul.txt, C, , Setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 04:30AM [get_matrix]: Exporting matrix data to (C:\Users\ymino\AppData\Local\Temp\5\tmpi8xcjo3x.txt, C, , Setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 2, False
INFO 04:30AM [get_matrix]: Exporting matrix data to (C:\Users\ymino\AppData\Local\Temp\5\tmprc71efgg.txt, C, , Setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 3, False
INFO 04:30AM [get_matrix]: Exporting matrix data to (C:\Users\ymino\AppData\Local\Temp\5\tmp2714yxc3.txt, C, , Setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 4, False


,ground_main_plane,pad_bot_Q1,pad_top_Q1,readout_connector_pad_Q1
ground_main_plane,177.75965,-44.75025,-38.34427,-37.00999
pad_bot_Q1,-44.75025,82.81507,-32.45477,-2.30888
pad_top_Q1,-38.34427,-32.45477,93.35391,-19.67194
readout_connector_pad_Q1,-37.00999,-2.30888,-19.67194,60.13047


### LOM Analysis
Now we provide the junction lumped element values, and complete the analysis by plotting the convergence. This is the same steps used in the `suggested` flow from tutorial 4.1.

In [29]:
c1.setup.junctions=Dict(Lj=12.31, Cj=2)
c1.setup.freq_readout = 7.0
c1.setup.freq_bus = []

c1.run_lom()
c1.lumped_oscillator_all

[1, 2] [3]
Predicted Values

Transmon Properties
f_Q 5.493624 [GHz]
EC 320.497723 [MHz]
EJ 13.273404 [GHz]
alpha -374.754362 [MHz]
dispersion 59.019708 [KHz]
Lq 12.305036 [nH]
Cq 60.437959 [fF]
T1 46.987145 [us]

**Coupling Properties**

tCqbus1 -7.535469 [fF]
gbus1_in_MHz -119.152862 [MHz]
χ_bus1 -3.825482 [MHz]
1/T1bus1 3387.201800 [Hz]
T1bus1 46.987145 [us]
Bus-Bus Couplings


,fQ,EC,EJ,alpha,dispersion,gbus,chi_in_MHz,χr MHz,gr MHz
1,5.859357,368.125565,13.273404,-437.042562,189.835326,[-121.83756770420214],[-7.291408804967165],7.291409,121.837568
2,5.840883,365.627263,13.273404,-433.731644,179.597972,[-112.81044513941916],[-6.048560188941762],6.048560,112.810445
3,5.75074,353.579503,13.273404,-417.833771,136.32109,[-115.0161625559102],[-5.378392760332066],5.378393,115.016163
4,5.671409,343.170959,13.273404,-404.189616,106.162812,[-115.32062725215741],[-4.7389164146794265],4.738916,115.320627
5,5.594864,333.29902,13.273404,-391.325604,82.835954,[-115.93484335569838],[-4.235792574451156],4.235793,115.934843
6,5.565138,329.510247,13.273404,-386.408168,75.085861,[-118.27807230946695],[-4.2077931475929775],4.207793,118.278072
7,5.527324,324.72676,13.273404,-380.215183,66.162768,[-118.62893645196495],[-3.992109008649076],3.992109,118.628936
8,5.51191,322.788502,13.273404,-377.710705,62.805353,[-119.50684884796532],[-3.956863333507348],3.956863,119.506849
9,5.493624,320.497723,13.273404,-374.754362,59.019708,[-119.15286242765927],[-3.825482151928471],3.825482,119.152862


And plot the convergence.

In [30]:
c1.plot_convergence();
c1.plot_convergence_chi()

  self._hfss_variables[variation] = pd.Series(



Design "TransmonQubit_q3d" info:
	# eigenmodes    0
	# variations    1


INFO 04:30AM [hfss_report_full_convergence]: Creating report for variation 0


Release the simulator and close the analysis.

In [31]:
c1.sim.close()

(optional) close the GUI.

In [32]:
# gui.main_window.close()